# Day 039 | Logistic Regression
## 100 Days of ML for Geotechnical Engineering
**Phase:** Classical Machine Learning

**Author:** Ripon Chandra Malo  
**Date:** 2026

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| **Logistic Regression** | Classification using a probability model | Predict liquefaction (yes/no) |
| **Sigmoid Function** | Maps any number to 0-1 probability | Convert soil parameters to liquefaction probability |
| **Decision Boundary** | Threshold where prediction flips | N-value below which liquefaction is likely |
| **Probability Output** | Model gives confidence, not just yes/no | "75% chance of liquefaction at this site" |
| **Confusion Matrix** | Table of correct and incorrect predictions | True positives, false negatives, etc. |
| **Precision vs Recall** | Tradeoff between false alarms and missed cases | Miss a liquefaction event vs false alarm |

---

## Our Geotechnical Problem

During an earthquake, loose saturated sands can lose their strength and behave like a liquid.
This is **liquefaction** — one of the most dangerous geotechnical hazards.

We have field data from sites that experienced earthquakes:
- **Features:** depth, SPT N-value, fines content, water table depth, peak ground acceleration
- **Target:** Did liquefaction occur? (1 = yes, 0 = no)

**Goal:** Build a classifier that predicts liquefaction risk from soil and seismic parameters.

---

## Part 1: Generate and Explore Liquefaction Data

We create a synthetic dataset based on established liquefaction correlations
(simplified Seed & Idriss approach).

---

In [ ]:
# ─────────────────────────────────────────────────────
# GENERATE SYNTHETIC LIQUEFACTION DATA
# ─────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
n = 120

# ── Generate realistic features ──
depth = np.random.uniform(1.5, 15.0, n)                    # meters
N_value = np.random.randint(3, 45, n)                       # SPT blow count
fines_content = np.random.uniform(0, 60, n)                 # percent
water_table_depth = np.random.uniform(0.5, 6.0, n)          # meters
peak_acceleration = np.random.uniform(0.05, 0.50, n)        # g

# ── Liquefaction model (simplified) ──
# Higher N-value -> less likely to liquefy
# Shallower water table -> more likely
# Higher acceleration -> more likely
# Higher fines -> slightly less likely (cohesion)
# Shallower depth -> more likely
logit = (
    -0.25 * N_value
    + 6.0 * peak_acceleration
    - 0.5 * water_table_depth
    - 0.02 * fines_content
    - 0.1 * depth
    + 3.5
    + np.random.normal(0, 1.0, n)  # noise
)
prob = 1 / (1 + np.exp(-logit))
liquefied = (prob > 0.5).astype(int)

df = pd.DataFrame({
    'depth': depth.round(1),
    'N_value': N_value,
    'fines_content': fines_content.round(1),
    'water_table_depth': water_table_depth.round(1),
    'peak_acceleration': peak_acceleration.round(3),
    'liquefied': liquefied
})

# ── Save for reproducibility ──
df.to_csv('liquefaction_classify.csv', index=False)

print("Liquefaction Classification Dataset")
print("=" * 55)
print(f"Shape: {df.shape}")
print(f"\nClass distribution:")
print(f"  Liquefied (1): {(df['liquefied'] == 1).sum()} ({(df['liquefied'] == 1).mean():.1%})")
print(f"  No liq.  (0): {(df['liquefied'] == 0).sum()} ({(df['liquefied'] == 0).mean():.1%})")
print(f"\nDescriptive Statistics:")
print(df.describe().round(2))

In [ ]:
# ─────────────────────────────────────────────────────
# VISUALIZE DATA BY CLASS
# ─────────────────────────────────────────────────────
features = ['depth', 'N_value', 'fines_content', 'water_table_depth', 'peak_acceleration']

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
colors = {0: '#4CAF50', 1: '#F44336'}
labels = {0: 'No Liquefaction', 1: 'Liquefied'}

for ax, feat in zip(axes, features):
    for cls in [0, 1]:
        mask = df['liquefied'] == cls
        ax.hist(df.loc[mask, feat], bins=15, alpha=0.6,
                color=colors[cls], label=labels[cls], edgecolor='black')
    ax.set_title(feat, fontweight='bold', fontsize=10)
    ax.set_xlabel(feat)
    ax.set_ylabel('Count')

axes[-1].legend(fontsize=8)
plt.suptitle('Feature Distributions by Liquefaction Class', fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('liquefaction_distributions.png', dpi=100, bbox_inches='tight')
plt.show()

print("Observations:")
print("  - Lower N_value sites are more likely to liquefy (loose sands)")
print("  - Higher peak acceleration increases liquefaction risk")
print("  - Shallow water table contributes to liquefaction")

In [ ]:
# ─────────────────────────────────────────────────────
# KEY SCATTER: N-value vs Peak Acceleration
# ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

for cls in [0, 1]:
    mask = df['liquefied'] == cls
    ax.scatter(df.loc[mask, 'peak_acceleration'], df.loc[mask, 'N_value'],
               c=colors[cls], label=labels[cls], s=50, alpha=0.7,
               edgecolors='black', linewidth=0.3)

ax.set_xlabel('Peak Ground Acceleration (g)', fontsize=12)
ax.set_ylabel('SPT N-value', fontsize=12)
ax.set_title('Liquefaction Triggering Chart\n(simplified Seed & Idriss style)',
             fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('triggering_chart.png', dpi=100, bbox_inches='tight')
plt.show()

print("This looks like the classic liquefaction triggering chart!")
print("Low N-value + High acceleration = liquefaction zone (red)")
print("High N-value + Low acceleration = safe zone (green)")
print("\nLogistic regression will learn this boundary automatically.")

## Part 2: The Sigmoid Function

Logistic regression uses the **sigmoid function** to convert any real number
into a probability between 0 and 1:

**sigma(z) = 1 / (1 + e^(-z))**

where z = w0 + w1*x1 + w2*x2 + ... (a linear combination of features)

---

In [ ]:
# ─────────────────────────────────────────────────────
# THE SIGMOID FUNCTION
# ─────────────────────────────────────────────────────
def sigmoid(z):
    """Convert logit to probability."""
    return 1 / (1 + np.exp(-z))

z = np.linspace(-8, 8, 200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Sigmoid curve ──
ax = axes[0]
ax.plot(z, sigmoid(z), 'b-', linewidth=3)
ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Threshold = 0.5')
ax.axvline(0, color='gray', linestyle=':', alpha=0.5)
ax.fill_between(z, sigmoid(z), 0.5, where=sigmoid(z) >= 0.5,
                alpha=0.1, color='red', label='Predict: Liquefied')
ax.fill_between(z, sigmoid(z), 0.5, where=sigmoid(z) < 0.5,
                alpha=0.1, color='green', label='Predict: Safe')
ax.set_xlabel('z (linear combination of features)', fontsize=11)
ax.set_ylabel('Probability', fontsize=11)
ax.set_title('The Sigmoid Function', fontweight='bold', fontsize=13)
ax.legend(fontsize=9)
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)

# ── N-value example ──
ax = axes[1]
# Simulate: probability of liquefaction vs N-value (holding others constant)
n_range = np.linspace(0, 40, 100)
# z = intercept - coeff * N_value
z_demo = 3.0 - 0.2 * n_range
prob_demo = sigmoid(z_demo)

ax.plot(n_range, prob_demo, 'b-', linewidth=3)
ax.axhline(0.5, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('SPT N-value', fontsize=11)
ax.set_ylabel('P(Liquefaction)', fontsize=11)
ax.set_title('Liquefaction Probability vs N-value\n(all other factors held constant)',
             fontweight='bold', fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)

# Mark the decision boundary
n_boundary = 3.0 / 0.2
ax.axvline(n_boundary, color='orange', linestyle='--', linewidth=2,
           label=f'Decision boundary (N={n_boundary:.0f})')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('sigmoid_function.png', dpi=100, bbox_inches='tight')
plt.show()

print("Key properties of the sigmoid:")
print("  - Output is always between 0 and 1 (valid probability)")
print("  - z = 0 gives probability = 0.5 (the decision boundary)")
print("  - z >> 0 gives probability near 1 (confident positive)")
print("  - z << 0 gives probability near 0 (confident negative)")

## Part 3: Train a Logistic Regression Model

Now we use sklearn to fit a logistic regression classifier on our liquefaction data.

---

In [ ]:
# ─────────────────────────────────────────────────────
# TRAIN LOGISTIC REGRESSION
# ─────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report, roc_curve, roc_auc_score)

# ── Prepare data ──
X = df[features].values
y = df['liquefied'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# ── Scale features ──
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# ── Fit logistic regression ──
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_s, y_train)

# ── Predictions ──
y_pred = lr.predict(X_test_s)
y_prob = lr.predict_proba(X_test_s)[:, 1]  # probability of class 1

print("Logistic Regression — Liquefaction Prediction")
print("=" * 55)
print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"AUC-ROC:  {roc_auc_score(y_test, y_prob):.3f}")

In [ ]:
# ─────────────────────────────────────────────────────
# MODEL COEFFICIENTS — What drives liquefaction?
# ─────────────────────────────────────────────────────
print("Model Coefficients (on scaled features)")
print("=" * 55)
print(f"Intercept: {lr.intercept_[0]:+.4f}")
print()

coef_df = pd.DataFrame({
    'Feature': features,
    'Coefficient': lr.coef_[0],
    'Abs_Coef': np.abs(lr.coef_[0])
}).sort_values('Abs_Coef', ascending=False)

for _, row in coef_df.iterrows():
    direction = 'increases' if row['Coefficient'] > 0 else 'decreases'
    print(f"  {row['Feature']:25s}  coef = {row['Coefficient']:+.4f}  ({direction} liq. risk)")

# ── Visualize ──
fig, ax = plt.subplots(figsize=(8, 4))
colors_bar = ['#F44336' if c > 0 else '#4CAF50' for c in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors_bar,
        edgecolor='black', alpha=0.8)
ax.set_xlabel('Coefficient (scaled)', fontsize=11)
ax.set_title('Feature Importance for Liquefaction Prediction', fontweight='bold', fontsize=12)
ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nRed = increases liquefaction risk, Green = decreases risk")

## Part 4: Confusion Matrix and Classification Metrics

For classification, accuracy alone is not enough. We need to understand:
- **True Positives (TP):** Correctly predicted liquefaction
- **False Negatives (FN):** Missed a liquefaction event (dangerous!)
- **False Positives (FP):** False alarm (costly but not dangerous)
- **True Negatives (TN):** Correctly predicted no liquefaction

---

In [ ]:
# ─────────────────────────────────────────────────────
# CONFUSION MATRIX
# ─────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Raw counts ──
ax = axes[0]
im = ax.imshow(cm, cmap='Blues', interpolation='nearest')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Safe', 'Liquefied'])
ax.set_yticklabels(['Safe', 'Liquefied'])
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('Actual', fontsize=11)
ax.set_title('Confusion Matrix (counts)', fontweight='bold')

# Annotate cells
labels_cm = [['TN', 'FP'], ['FN', 'TP']]
for i in range(2):
    for j in range(2):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, f'{labels_cm[i][j]}\n{cm[i, j]}',
                ha='center', va='center', fontsize=14, fontweight='bold', color=color)

# ── Classification report as text ──
ax = axes[1]
report = classification_report(y_test, y_pred, target_names=['Safe', 'Liquefied'])
ax.text(0.1, 0.5, report, fontsize=12, family='monospace',
        verticalalignment='center', transform=ax.transAxes)
ax.set_title('Classification Report', fontweight='bold')
ax.axis('off')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"Detailed Metrics:")
print(f"  Accuracy:  {(tp+tn)/(tp+tn+fp+fn):.3f}  (overall correctness)")
print(f"  Precision: {tp/(tp+fp):.3f}  (of predicted liquefied, how many actually were)")
print(f"  Recall:    {tp/(tp+fn):.3f}  (of actual liquefied, how many did we catch)")
print(f"  F1-score:  {2*tp/(2*tp+fp+fn):.3f}  (harmonic mean of precision and recall)")
print(f"\nIn geotechnical safety:")
print(f"  False Negatives ({fn}) are DANGEROUS — we missed real liquefaction events")
print(f"  False Positives ({fp}) are COSTLY — unnecessary ground improvement")

## Part 5: Probability Output and Decision Threshold

Logistic regression outputs **probabilities**, not just labels. We can adjust the
decision threshold to trade off precision and recall.

In safety-critical applications like liquefaction:
- Lower threshold = catch more true cases (higher recall) at the cost of more false alarms
- Higher threshold = fewer false alarms (higher precision) but may miss real events

---

In [ ]:
# ─────────────────────────────────────────────────────
# PROBABILITY DISTRIBUTION
# ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Histogram of predicted probabilities ──
ax = axes[0]
for cls, color, label in [(0, '#4CAF50', 'Actually Safe'), (1, '#F44336', 'Actually Liquefied')]:
    mask = y_test == cls
    ax.hist(y_prob[mask], bins=15, alpha=0.6, color=color, label=label, edgecolor='black')
ax.axvline(0.5, color='black', linestyle='--', linewidth=2, label='Threshold = 0.5')
ax.set_xlabel('Predicted Probability of Liquefaction', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Predicted Probability Distribution', fontweight='bold', fontsize=12)
ax.legend(fontsize=9)

# ── Effect of threshold on precision/recall ──
ax = axes[1]
thresholds = np.linspace(0.1, 0.9, 50)
precisions = []
recalls = []

for t in thresholds:
    y_t = (y_prob >= t).astype(int)
    tp_t = ((y_t == 1) & (y_test == 1)).sum()
    fp_t = ((y_t == 1) & (y_test == 0)).sum()
    fn_t = ((y_t == 0) & (y_test == 1)).sum()
    precisions.append(tp_t / (tp_t + fp_t) if (tp_t + fp_t) > 0 else 0)
    recalls.append(tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0)

ax.plot(thresholds, precisions, 'b-', linewidth=2, label='Precision')
ax.plot(thresholds, recalls, 'r-', linewidth=2, label='Recall')
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='Default threshold')
ax.set_xlabel('Decision Threshold', fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Precision vs Recall at Different Thresholds', fontweight='bold', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('probability_threshold.png', dpi=100, bbox_inches='tight')
plt.show()

print("For safety-critical liquefaction assessment:")
print("  Use a LOWER threshold (e.g., 0.3) to catch more potential liquefaction cases")
print("  Accept more false alarms to avoid missing dangerous sites")

In [ ]:
# ─────────────────────────────────────────────────────
# ROC CURVE — Overall discrimination ability
# ─────────────────────────────────────────────────────
fpr, tpr, roc_thresholds = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, 'b-', linewidth=3, label=f'Logistic Regression (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random classifier')
ax.fill_between(fpr, tpr, alpha=0.1, color='blue')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — Liquefaction Classifier', fontweight='bold', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)

plt.tight_layout()
plt.savefig('roc_curve.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"AUC = {auc:.3f}")
print("  AUC = 0.5: random guessing (useless)")
print("  AUC = 1.0: perfect classification")
print(f"  AUC = {auc:.3f}: {'good' if auc > 0.8 else 'moderate'} discrimination ability")

## Part 6: Decision Boundary Visualization

Let us visualize the decision boundary using the two most important features.

---

In [ ]:
# ─────────────────────────────────────────────────────
# DECISION BOUNDARY (2D slice: N_value vs peak_acceleration)
# ─────────────────────────────────────────────────────
# Train a 2-feature model for visualization
X_2d = df[['N_value', 'peak_acceleration']].values
scaler_2d = StandardScaler()
X_2d_s = scaler_2d.fit_transform(X_2d)

lr_2d = LogisticRegression(random_state=42)
lr_2d.fit(X_2d_s, y)

# ── Create mesh grid ──
xx, yy = np.meshgrid(
    np.linspace(X_2d[:, 0].min() - 2, X_2d[:, 0].max() + 2, 200),
    np.linspace(X_2d[:, 1].min() - 0.05, X_2d[:, 1].max() + 0.05, 200)
)
mesh_points = np.c_[xx.ravel(), yy.ravel()]
mesh_scaled = scaler_2d.transform(mesh_points)
Z = lr_2d.predict_proba(mesh_scaled)[:, 1].reshape(xx.shape)

fig, ax = plt.subplots(figsize=(10, 7))
contour = ax.contourf(xx, yy, Z, levels=20, cmap='RdYlGn_r', alpha=0.6)
plt.colorbar(contour, ax=ax, label='P(Liquefaction)')

# ── Plot decision boundary ──
ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2, linestyles='--')

# ── Plot data points ──
for cls in [0, 1]:
    mask = y == cls
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               c=colors[cls], label=labels[cls], s=50, alpha=0.8,
               edgecolors='black', linewidth=0.5)

ax.set_xlabel('SPT N-value', fontsize=12)
ax.set_ylabel('Peak Ground Acceleration (g)', fontsize=12)
ax.set_title('Logistic Regression Decision Boundary\n(dashed line = 50% probability)',
             fontweight='bold', fontsize=13)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('decision_boundary.png', dpi=100, bbox_inches='tight')
plt.show()

print("The decision boundary is a straight line in feature space.")
print("Logistic regression finds the linear separator that best divides the classes.")
print("Points near the boundary have probabilities close to 0.5 (uncertain predictions).")

## Part 7: Making Predictions on New Sites

Let us use our trained model to assess liquefaction risk at hypothetical new sites.

---

In [ ]:
# ─────────────────────────────────────────────────────
# PREDICT LIQUEFACTION RISK AT NEW SITES
# ─────────────────────────────────────────────────────
new_sites = pd.DataFrame({
    'depth': [5.0, 8.0, 3.0, 12.0],
    'N_value': [8, 25, 5, 35],
    'fines_content': [15, 5, 40, 10],
    'water_table_depth': [1.5, 4.0, 1.0, 5.0],
    'peak_acceleration': [0.30, 0.15, 0.40, 0.10]
})

# ── Predict ──
new_scaled = scaler.transform(new_sites.values)
predictions = lr.predict(new_scaled)
probabilities = lr.predict_proba(new_scaled)[:, 1]

print("Liquefaction Risk Assessment — New Sites")
print("=" * 65)
for i, (_, row) in enumerate(new_sites.iterrows()):
    risk = 'HIGH RISK' if probabilities[i] > 0.7 else ('MODERATE' if probabilities[i] > 0.3 else 'LOW RISK')
    result = 'Liquefied' if predictions[i] == 1 else 'Safe'
    print(f"\nSite {i+1}: depth={row['depth']}m, N={row['N_value']}, "
          f"PGA={row['peak_acceleration']}g")
    print(f"  Prediction: {result}  |  P(liq) = {probabilities[i]:.1%}  |  {risk}")

---

## Practice Exercises

### Exercise 1: Safety Threshold
Lower the decision threshold to 0.3 instead of 0.5. Recompute the confusion matrix.
How does recall change? How does precision change? Is this better for safety?

### Exercise 2: Feature Importance
Remove `fines_content` (the least important feature) and retrain the model.
Does accuracy change? What does this tell you about feature selection?

### Exercise 3: Multiclass Extension
Create a 3-class target: "No liquefaction" (prob < 0.3), "Marginal" (0.3-0.7),
"Liquefied" (prob > 0.7). Train a multiclass logistic regression. How does the
confusion matrix look?

---

In [ ]:
# ─────────────────────────────────────────────────────
# EXERCISE 1: Safety threshold
# ─────────────────────────────────────────────────────
# threshold = 0.3
# y_pred_safe = (y_prob >= threshold).astype(int)
# cm_safe = confusion_matrix(y_test, y_pred_safe)
# print(f"Threshold = {threshold}")
# print(f"Confusion matrix:\n{cm_safe}")
# print(classification_report(y_test, y_pred_safe, target_names=['Safe', 'Liquefied']))

In [ ]:
# ─────────────────────────────────────────────────────
# EXERCISE 2: Feature removal
# ─────────────────────────────────────────────────────
# features_reduced = ['depth', 'N_value', 'water_table_depth', 'peak_acceleration']
# X_red = df[features_reduced].values
# X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
#     X_red, y, test_size=0.25, random_state=42, stratify=y)
# scaler_r = StandardScaler()
# lr_r = LogisticRegression(random_state=42)
# lr_r.fit(scaler_r.fit_transform(X_train_r), y_train_r)
# acc = accuracy_score(y_test_r, lr_r.predict(scaler_r.transform(X_test_r)))
# print(f"Accuracy without fines_content: {acc:.3f}")

---

## Key Takeaways

1. **Logistic regression is for classification, not regression.** Despite the name, it predicts categories (liquefied vs safe) by modeling the probability of each class.

2. **The sigmoid function maps features to probabilities.** Any combination of soil parameters gets mapped to a number between 0 and 1 representing liquefaction likelihood.

3. **The decision threshold is not fixed at 0.5.** In safety-critical geotechnical applications, lower the threshold to catch more true positives (higher recall) even if it means more false alarms.

4. **Precision vs recall is a fundamental tradeoff.** Missing a liquefaction event (false negative) is far more dangerous than a false alarm (false positive). Optimize for recall in safety applications.

5. **Model coefficients reveal feature importance.** N-value and peak acceleration are the strongest predictors of liquefaction, which aligns with established geotechnical knowledge.

6. **AUC-ROC measures overall discrimination.** An AUC above 0.8 means the model distinguishes liquefied from non-liquefied sites well across all thresholds.

---

## How This Connects to ML

| What We Did Today | What It Becomes in ML |
|---|---|
| Logistic regression | Foundation for neural network output layers |
| Sigmoid function | Activation function in deep learning |
| Confusion matrix | Standard evaluation for any classifier |
| Precision/Recall | Choosing metrics that match business needs |
| ROC curve | Comparing multiple models objectively |
| Decision boundary | Understanding model geometry (linear vs nonlinear) |
| Probability calibration | Reliable risk assessment in practice |

---

*Day 039 of 100 — Keep going!*